In [1]:
import pandas as pd
import sklearn 
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('wdi_poverty_panel_raw.csv')

In [3]:
df.head()

,Country Name,Country Code,Series Name,Series Code,2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],...,2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022]
0,Afghanistan,AFG,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,28.6000011706788,8.83227780288267,1.41411799339429,11.2297148272859,5.35740324748592,13.8263195471281,...,5.60074465863221,2.72454336219565,1.45131466066431,2.26031420279821,2.6470032027451,1.18922812944517,3.91160341625552,-2.35110067203466,-20.7388393676343,-6.24017199240269
1,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,..,..,..,12.6862687216715,6.78459655001655,8.68057078513406,...,7.38577178397855,4.67399603536345,-0.661709164713742,4.38389195513915,4.97595150553833,0.626149149168847,2.30237251516834,5.60188791482224,5.13320340824963,13.7121023720065
2,Afghanistan,AFG,"Life expectancy at birth, total (years)",SP.DYN.LE00.IN,56.225,57.171,57.81,58.247,58.553,58.956,...,62.188,62.26,62.27,62.646,62.406,62.443,62.941,61.454,60.417,65.617
3,Afghanistan,AFG,"Government expenditure on education, total (% ...",SE.XPD.TOTL.GD.ZS,..,..,..,..,..,..,...,3.45445990562439,3.69521999359131,3.25578999519348,4.5439600944519,4.34319019317627,..,..,..,..,..
4,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,0.00456139517022146,0.0878912528559713,0.105809030021958,1.22414808372471,2.10712364546412,1.899999976,...,5.900000095,7,8.260000229,11,13.5,16.79999924,17.60000038,17.04850006,16.51429939,15.86629963


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Country Name   3040 non-null   object
 1   Country Code   3038 non-null   object
 2   Series Name    3038 non-null   object
 3   Series Code    3038 non-null   object
 4   2002 [YR2002]  3038 non-null   object
 5   2003 [YR2003]  3038 non-null   object
 6   2004 [YR2004]  3038 non-null   object
 7   2005 [YR2005]  3038 non-null   object
 8   2006 [YR2006]  3038 non-null   object
 9   2007 [YR2007]  3038 non-null   object
 10  2008 [YR2008]  3038 non-null   object
 11  2009 [YR2009]  3038 non-null   object
 12  2010 [YR2010]  3038 non-null   object
 13  2011 [YR2011]  3038 non-null   object
 14  2012 [YR2012]  3038 non-null   object
 15  2013 [YR2013]  3038 non-null   object
 16  2014 [YR2014]  3038 non-null   object
 17  2015 [YR2015]  3038 non-null   object
 18  2016 [YR2016]  3038 non-null

In [5]:
df = df.dropna(subset=['Country Name'])

In [6]:
year_cols = [c for c in df.columns if c.startswith('20')]

In [7]:
long_df = df.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

In [8]:
long_df['Year'] = long_df['Year'].str.extract(r'(\d{4})').astype(int)
long_df['Value'] = pd.to_numeric(long_df['Value'], errors='coerce')  # '..' becomes NaN


In [9]:
panel_df = long_df.pivot_table(
    index=['Country Name', 'Country Code', 'Year'],
    columns='Series Name',
    values='Value'
).reset_index()

In [10]:
panel_df.shape

(4557, 17)

In [11]:
target_col = 'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)'

In [12]:
panel_df = panel_df.dropna(subset=[target_col])

In [13]:
predictor_cols = [c for c in panel_df.columns if c not in ['Country Name', 'Country Code', 'Year', target_col]]

In [14]:
panel_df[predictor_cols] = panel_df[predictor_cols].fillna(panel_df[predictor_cols].median())

In [15]:
rename_map = {
    'Access to electricity (% of population)': 'electricity_access',
    'Agriculture, forestry, and fishing, value added (% of GDP)': 'agri_value_added',
    'Current health expenditure (% of GDP)': 'health_expenditure',
    'Foreign direct investment, net inflows (% of GDP)': 'fdi_inflows',
    'GDP growth (annual %)': 'gdp_growth',
    'GDP per capita (current US$)': 'gdp_per_capita',
    'Government expenditure on education, total (% of GDP)': 'education_expenditure',
    'Individuals using the Internet (% of population)': 'internet_users',
    'Inflation, consumer prices (annual %)': 'inflation',
    'Life expectancy at birth, total (years)': 'life_expectancy',
    'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)': 'poverty_ratio',
    'School enrollment, primary (% gross)': 'school_enrollment',
    'Unemployment, total (% of total labor force) (national estimate)': 'unemployment',
    'Urban population (% of total population)': 'urban_population',
}



In [16]:
panel_df = panel_df.rename(columns=rename_map)
panel_df.columns = [c.lower().replace(' ', '_') for c in panel_df.columns]  

In [17]:

panel_df.to_csv('wdi_poverty_panel_clean.csv', index=False)